In [46]:
import pandas as pd
import numpy as np

df  = pd.read_excel("basedados.xlsx")

dataframe_dados_clientes = df.iloc[: , 1:19] #pega todas as linhas da coluna da 1 a 18
dataframe_gabarito = df.iloc[: , 19] #pega todas as linhas da coluna 20 (gabarito)

array_dados_clientes = dataframe_dados_clientes.values
array_gabarito = dataframe_gabarito.values


In [47]:
def criar_cromossomos(qtd_cromossomos: int=6, qtd_genes: int = 19) -> np.ndarray:
    """
    Cria uma matriz onde cada linha e um cromossomo e cada coluna e um gene
    """
    # Passamos as duas dimensões para o rand: linhas (indivíduos) e colunas (genes)
    array_cromossomos = -1 + 2 * np.random.rand(qtd_cromossomos, qtd_genes)
    
    return array_cromossomos

In [48]:
cromossomos = criar_cromossomos()

In [49]:
print(cromossomos)

[[ 0.20350133  0.77687222  0.3946868  -0.78377684  0.3311628  -0.54666587
  -0.94703034  0.38136671 -0.90226907  0.42384132 -0.03985815  0.53324974
  -0.55268296 -0.73022355  0.94237262 -0.22693094 -0.51834881  0.02060917
   0.39422918]
 [-0.35547225  0.65041344 -0.38074145  0.97324868  0.84035084 -0.74346044
   0.02319926 -0.19143403 -0.28268917  0.1066586  -0.05733885  0.18185623
   0.6804167  -0.78163364  0.73192424 -0.42648486 -0.13700682 -0.99108795
  -0.40808548]
 [-0.94079402 -0.60392443 -0.04339646  0.24673013 -0.73959123  0.69360974
  -0.76094415  0.38657002  0.23939582  0.20728902 -0.23719564 -0.92335757
   0.25581132 -0.75305855 -0.25680786 -0.64600738 -0.92358379 -0.86754774
  -0.60537408]
 [-0.76998814  0.90774059 -0.30027977 -0.80751585  0.68009309 -0.60178414
  -0.27134417 -0.90778809 -0.06574257 -0.06746318 -0.89057058  0.58676172
   0.14277517 -0.87288854 -0.34103325 -0.79259452  0.49007227  0.13827156
   0.28470043]
 [ 0.66447613 -0.11843075 -0.78947787 -0.65183532 -0

In [50]:
print(cromossomos[0,0])

0.203501325422208


In [51]:
def calcular_fitness(cromossomos: np.ndarray, array_dados_clientes: np.ndarray, array_gabarito: np.ndarray) -> np.ndarray:
    """
    Pega cada linha do array_cromossomos (menos o primeiro termo) e itera sobre cada linha da array de dados clientes
    cada linha (iteracao) e um produto escalar
    cada iteracao vai gerar um vetor coluna contendo 0 e 1 chamado de vetor hipotese

    depois comparar o vetor hipotese de cada cromossomo com o vetor gabarito e calcular a porcentagem de acerto

    depois calcular o fitness seguindo a formula 

    percentual_adimplente = quantidade de 1 no vetor hipotese/ total de 1 no gabarito
    percentual_inadimplente - quantidade de 0 no vetor hipotese/ total de 0 no gabarito

    fitness = percentual_adimplente * percentual_inadimplente

    cada cromossomo vai ter um fitness. Somar todos os fitness e calcular a porcentagem relativa de acerta de cada cromossomo
    com base nisso
    """

    total_adimplentes = np.sum(array_gabarito == 1)
    total_inadimplentes = np.sum(array_gabarito == 0)
    
    lista_hipotese = []

    #iteracao pra cada cromossomo
    for linha in cromossomos:  # o for numa array vai de linha em linha automaticamente
        
        bias = linha[0]
        genes = linha[1:]

        q = np.dot(array_dados_clientes, genes) + bias

        #claudio ajudou, onde cada elemento de Q for maior igual a zero troque por 1, se nao troque por zero
        vetor_hipotese = np.where(q >= 0, 1, 0)

        acertos_adimplentes = np.sum((vetor_hipotese == 1) & (array_gabarito == 1)) #se a hipotese for 1 e o gabarito for 1 ele acertou, entao contabiliza
        acertos_inadimplentes = np.sum((vetor_hipotese == 0) & (array_gabarito == 0)) #se a hipotese for 0 e o gabarito for 0 ele acertou, entao contabiliza
        # se nao o & da false e ele nao soma 


        percentual_adimplente = acertos_adimplentes / total_adimplentes
        percentual_inadimplente = acertos_inadimplentes / total_inadimplentes

        fitness = percentual_adimplente * percentual_inadimplente
        
        lista_hipotese.append(fitness)

    
    return np.array(lista_hipotese) #array com o fitness de cada cromossomo

In [52]:
vetor_fitnesses = calcular_fitness(cromossomos, array_dados_clientes, array_gabarito)
print(vetor_fitnesses)

[0.52631579 0.0877193  0.         0.         0.         0.20175439]


In [53]:
def fitness_percentual(vetor_fitnesses: np.ndarray) -> np.ndarray:
    """
    Recebe um array de fitnesses e retorna um array de fitnesses percentualizados.
    """
    soma_fitness = np.sum(vetor_fitnesses)
    percentual_fitness = vetor_fitnesses / soma_fitness
    return percentual_fitness

In [59]:
percentual_fitnesses = fitness_percentual(vetor_fitnesses)
print(percentual_fitnesses)

[0.64516129 0.10752688 0.         0.         0.         0.24731183]


In [64]:
percentual_fitnesses[0]

np.float64(0.6451612903225807)

In [75]:

i = 0
while i < len(cromossomos):
    bloco = percentual_fitnesses[i]
    print(bloco)
    i = i+1

0.6451612903225807
0.10752688172043012
0.0
0.0
0.0
0.24731182795698928


In [ ]:
def selecionar_pais_roleta(cromossomos: np.ndarray, percentual_fitnesses: np.ndarray):
    """
    gera dois numeros aleatorios entre 0 e 1 pra escolher quem serao os pais
    """

    #separacao

    for cromossomo in len(cromossomos):
        i = 0
        bloco

    



    random_pai = np.random.rand()
    random_mae = np.random.rand()

In [ ]:
def selecionar_pais_roleta(cromossomos: np.ndarray, percentual_fitnesses: np.ndarray):
    """
    gera dois numeros aleatorios entre 0 e 1 pra escolher quem serao os pais
    """
    # 1. Cria as fronteiras da roleta usando a soma acumulada
    roleta_acumulada = np.cumsum(percentual_fitnesses)
    
    # 2. Gera dois números aleatórios independentes entre 0 e 1
    r_pai, r_mae = np.random.rand(2)
    
    # 3. Descobre matematicamente em qual fatia os números caíram
    indice_pai = np.searchsorted(roleta_acumulada, r_pai)
    indice_mae = np.searchsorted(roleta_acumulada, r_mae)
    
    # 4. Extrai as linhas correspondentes da sua matriz de população
    pai = cromossomos[indice_pai]
    mae = cromossomos[indice_mae]
    
    # Opcional: Imprime os índices para você acompanhar o sorteio no notebook
    print(f"Giro da Roleta -> Pai: Índice {indice_pai} (Sorteio: {r_pai:.4f}) | Mãe: Índice {indice_mae} (Sorteio: {r_mae:.4f})")
    
    return pai, mae

In [56]:
pai, mae = selecionar_pais_roleta(cromossomos, percentual_fitnesses)

Giro da Roleta -> Pai: Índice 0 (Sorteio: 0.1918) | Mãe: Índice 0 (Sorteio: 0.2478)


In [57]:
def crossover(pai: np.ndarray, mae: np.ndarray) -> (np.ndarray, np.ndarray, np.ndarray):
    """
    Realiza o crossover entre dois cromossomos (pai e mãe) para gerar tres filhos.
    O crossover é feito escolhendo um ponto de corte aleatório e combinando os genes dos pais.
    """
    # 1. Escolhe um ponto de corte aleatório (entre 1 e o número de genes - 1)
    ponto_corte = np.random.randint(1, len(pai) - 1)
    
    # 2. Cria os filhos combinando os genes dos pais
    filho1 = np.concatenate((pai[:ponto_corte], mae[ponto_corte:]))
    filho2 = np.concatenate((mae[:ponto_corte], pai[ponto_corte:]))
    
    # Opcional: Imprime o ponto de corte e os filhos para acompanhar no notebook
    print(f"Crossover -> Ponto de Corte: {ponto_corte} | Filho 1: {filho1} | Filho 2: {filho2}")
    
    return filho1, filho2

In [58]:
print(crossover(pai,mae))

Crossover -> Ponto de Corte: 10 | Filho 1: [ 0.20350133  0.77687222  0.3946868  -0.78377684  0.3311628  -0.54666587
 -0.94703034  0.38136671 -0.90226907  0.42384132 -0.03985815  0.53324974
 -0.55268296 -0.73022355  0.94237262 -0.22693094 -0.51834881  0.02060917
  0.39422918] | Filho 2: [ 0.20350133  0.77687222  0.3946868  -0.78377684  0.3311628  -0.54666587
 -0.94703034  0.38136671 -0.90226907  0.42384132 -0.03985815  0.53324974
 -0.55268296 -0.73022355  0.94237262 -0.22693094 -0.51834881  0.02060917
  0.39422918]
(array([ 0.20350133,  0.77687222,  0.3946868 , -0.78377684,  0.3311628 ,
       -0.54666587, -0.94703034,  0.38136671, -0.90226907,  0.42384132,
       -0.03985815,  0.53324974, -0.55268296, -0.73022355,  0.94237262,
       -0.22693094, -0.51834881,  0.02060917,  0.39422918]), array([ 0.20350133,  0.77687222,  0.3946868 , -0.78377684,  0.3311628 ,
       -0.54666587, -0.94703034,  0.38136671, -0.90226907,  0.42384132,
       -0.03985815,  0.53324974, -0.55268296, -0.73022355,